# Demo of Framework execution

This notebook shows the steps of the Framework related to the evaluation of the risk matrix.
Given a system to analyse, the outcome is a mathematical object which contains information on the risk which the system is exposed to. The drivers within the hazard, exposure and vulnerability determinants regarding the system are quantified through indicators and are combined to obtain a numeric characterisation of the risk.

The system is fixed, hence numeric information about exposure and vulnerability are fixed throughout the analysis. 
Ensembles of the resulting mathematical objects are created varying indicators of the hazard drivers and climate data used to evaluate them.

The content and the Python code are general: they are valid for any choice of drivers, their indicators and data.
To show in practice how to execute this part of the Framework, a fictional risk assessment is performed. The example is described through text in *italic* and blocks of code introduced by the comment `# Example`.

## Step 1

Given the system to analyse, select drivers within the hazard determinant which could affect the system.

Note that at this point no reference to the componenents of the system, i.e. the assets, is made.
In general this distinction is not necessary for the part of the Framework regarding the hazard determinant. However, thinking to the system in terms of its components could help the analyst to identify the drivers affecting the system in its entirety.

*The system is Rome Ciampino airport. It is found to be affected only by drivers "heat wave" and "heavy precipitation". They are chosen for the next steps of the analysis.*

In [ ]:
# Example: insert drivers in Python code.

drivers_kw = ["heat_wave", "heavy_precipitation"]

## Step 2

Choose the indicators suitable to describe each driver.

*Indicators already provided by the xclim Python package are chosen for ease. [Their list](https://xclim.readthedocs.io/en/stable/indicators.html) is available in xclim official documentation, together with [a guide](https://xclim.readthedocs.io/en/stable/notebooks/extendxclim.html#Defining-new-indicators) on how to build new indicators.*  
*The chosen xclim indicators of heat wave are: [heat_wave_frequency](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_frequency), [heat_wave_max_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_max_length), [heat_wave_total_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_total_length) and [tx_max](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.tx_max).*  
*The chosen xclim indicators of heavy precipitation are: [max_n_day_precipitation_amount](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.max_n_day_precipitation_amount), [wetdays](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.wetdays) and [cwd](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.cwd). Indicator max_n_day_precipitation_amount is considered with two different window sizes.*

## Step 3

Implement the indicators in Python language.

One option is to load indicators as Python objects from a YAML file. Further details are available in [the specific section](https://xclim.readthedocs.io/en/stable/api.html#yaml-file-structure) of xclim official documentation.

In [ ]:
import pathlib

import xclim

In [ ]:
indicators = {}
for driver_kw in drivers_kw:
    _module = xclim.core.indicator.build_indicator_module_from_yaml(pathlib.Path(f"{driver_kw}.yml"))
    indicators.update(dict(_module.iter_indicators()))

## Step 4

Select climate data. Choose spatial boundaries to describe the local climate surrounding the system with sufficient accuracy and select the baseline period.
The selected data must contain the climate variables needed to compute the indicators.

Some preprocessing may be performed on data once loaded in Python. For instance, different datasets may refer to the same climate variable with different names and they need to be renamed to the same string to make the following Python code work with no user supervision.
For variables names [the xclim conventions](https://xclim.readthedocs.io/en/stable/readme.html#conventions) are followed. In particular, the available climate variables available natively in xclim are stored in `xclim.core.utils.VARIABLES`.

*Climate data used in the example are from ERA5-Land by ECMWF. The selected baseline period is 1980-2010. Two different datasets serve as example for the creation of an ensemble by varying climate data.*
*The selected indicators need variables "tasmin", "tasmax" and "pr" as inputs. First two variables are derived from variable "tas", while variable "pr" is derived from the precipitation amount, which is represented here with variable "tp". Hence data for climate variables "tas" and "tp" are sufficient for the following calculations.*
*Data are further restricted spatially to lower the computing time of indicators. A small region around Ciampino airport is selected, which coordinates of the central point taken from [Italian Civil Aviation Authority (ENAC)](https://www.enac.gov.it/node/35353).*

In [ ]:
import dask.distributed
import xarray

import utilities

try:
    print(cluster.dashboard_link)
except NameError:
    cluster = dask.distributed.LocalCluster(n_workers=8)
    client = dask.distributed.Client(cluster)

In [ ]:
# Example: choose datasets. To improve data management, each dataset is stored
# in a folder with same name and split in a folder for each climate variable.

datasets_kw = ["era5-land"]
variables_kw = ["2m_temperature", "total_precipitation"]
baseline_period = slice("1980", "2010")

dir_data = pathlib.Path(utilities.DIR_ROOT, "data")

In [ ]:
ds_hourly = {}
for dataset_kw in datasets_kw:
    # Load different variables of the same dataset in the same xarray Dataset.
    ds_hourly[dataset_kw] = xarray.Dataset()
    for variable_kw in variables_kw:
        path_data = pathlib.Path(dir_data, dataset_kw, variable_kw)
        _dataset = xarray.open_mfdataset(path_data.iterdir())
        ds_hourly[dataset_kw] = xarray.merge(
            [ds_hourly[dataset_kw], _dataset],
            combine_attrs="drop_conflicts"
        )
    # Restrict data to the selected baseline period.
    ds_hourly[dataset_kw] = ds_hourly[dataset_kw].sel(time=baseline_period, drop=True)

In [ ]:
# Example: fill missing values in ERA5Land dataset.

ds_hourly["era5-land"] = ds_hourly["era5-land"].sel(expver=1).combine_first(ds_hourly["era5-land"].sel(expver=5))

In [ ]:
# Example: set same names to same climate variables in every dataset.

ds_hourly["era5-land"] = ds_hourly["era5-land"].rename({
    "t2m": "tas"
})

In [ ]:
# Example: set CF Standard Name where missing, needed for automatic conversions.

ds_hourly["era5-land"]["tp"].attrs["standard_name"] = "lwe_thickness_of_precipitation_amount"

In [ ]:
# Example: restrict spatial extension of data.

from braun.utils.conversions import dms2dd

longitude_dms = (12, 35, 49.5700)  # Longitude east.
latitude_dms = (41, 47, 58.1500)  # Latitude north.

longitude_dd = dms2dd(*longitude_dms)
latitude_dd = dms2dd(*latitude_dms)

tolerance = 3 * 0.1  # Use era5-land dataset resolution.
ds_hourly["era5-land"] = ds_hourly["era5-land"].sel(
    longitude=slice(longitude_dd - tolerance, longitude_dd + tolerance),
    latitude=slice(latitude_dd + tolerance, latitude_dd - tolerance)
)

In [ ]:
# Example: evaluate derived climate variables needed as inputs of indicators.

# Use new xarray Dataset objects when introducing a new sampling frequency
# to have consistent time units.
ds_daily = {}
for dataset_kw in datasets_kw:
    ds_daily[dataset_kw] = xarray.Dataset()
    # Evaluate tasmin and tasmax.
    _resample = ds_hourly[dataset_kw]["tas"].resample(time="1D")
    ds_daily[dataset_kw]["tasmin"] = _resample.min()
    ds_daily[dataset_kw]["tasmax"] = _resample.max()
    # Resample tp at the sampling frequency needed to obtain pr.
    _resample = ds_hourly[dataset_kw]["tp"].resample(time="1D")
    ds_daily[dataset_kw]["tp"] = _resample.sum()

In [ ]:
# Example: evaluate pr, this conversion can not be done automatically.

for dataset_kw in datasets_kw:
    ds_daily[dataset_kw]["pr"] = xclim.core.units.convert_units_to(
        source=ds_daily[dataset_kw]["tp"],
        target="mmday",
        context="hydro"
    )

## Step 5

Evaluate indicators on each dataset.
The outcome is an ensemble of mathematical objects, each being a collection of yearly averages of the evaluated indicators. There is a member of the ensemble for each dataset.

*Only two datasets are used to build the ensemble, hence resulting statistical analysis is not accurate. The purpose of this example is limited to show how the Framework works, not to draw effective conclusions on its application.*

In [ ]:
# Indicators are evaluated separately for each dataset
# because data could be associated to different geographic coordinate systems.
# If this problem does not apply, then all datasets can be stored in the same
# xarray Dataset object with an additional dimension identifying each of them.
ds_member = {}
for dataset_kw in datasets_kw:
    ds_member[dataset_kw] = xarray.Dataset()
    for indicator_kw, indicator in indicators.items():
        ds_member[dataset_kw][indicator_kw] = indicator(ds=ds_daily[dataset_kw])

## Step 6

Create ensemble by varying the parameters of selected indicators.

With this procedure, an ensemble for each dataset is created.
Note that, given a dataset, the outcome of step 5 is only one member of the ensemble that is created in this step.

From an implementation point of view, in this step an xarray Dataset object is created for each indicator. This choice is made to avoid sparse matrices, an issue related to how xarray Dataset objects are structured: in general each indicator has different parameters and each parameter can assume different values (e.g. two indicators both rely on parameter `thresh_tasmin`, but the number of values chosen as arguments for each of them is different), hence if all combinations of these values were inserted in a single xarray Dataset object, all combined as coordinates in the dimension of the corresponding parameter, the resulting object would contain a huge number of empty cells (i.e. NaN values), wasting computing resources needlessly.
The empty cells are positioned at coordinates which are not valid arguments for the indicators (e.g. the xclim [`tx_max`](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.tx_max) indicator expects parameter `tasmax` but not parameter `thresh_tasmax`, hence any value of the latter would be an invalid coordinate for the outcomes of this indicator).

In this step, more in detail:
1. Identify the parameters which can be varied for each indicator. 
2. For each parameter decide a set of values that it can take. These sets can be intervals, which ideally are continuous, hence to allow computations on them they are first discretised into sequences of values, separated by a given resolution, with the endpoints included. In the implementation, the number of values in the interval is requested instead of the resolution for increased precision.
3. Evaluate each indicator for every combination of its arguments (i.e. the values of its parameters). The outcomes are inserted in an xarray Dataset object for each indicator, with the varied parameters as additional dimensions and the parameter values as coordinates.

*Same values are used for same parameters in different indicators (e.g. use same sequence of values for `thresh_tasmin` and `thresh_tasmax` parameters in every indicator that uses them). The same number of values is considered to check easily the number of operations. Moreover to speed up the computations, they are performed only for one indicator, hence a single xarray Dataset object results from this step.*

In [ ]:
from braun.packages.xclim import get_variable_parameters

In [ ]:
parameters = {}
# Prepare the storage for the xclim.DataArray objects
# which will be the additional coordinates.
for indicator_kw, indicator in indicators.items():
    parameters[indicator_kw] = get_variable_parameters(indicator)

In [ ]:
import numpy

In [ ]:
# Example: same values are used for same parameters in different indicators.

intervals = {}

interval_kw = "thresh_tasmin"
intervals[interval_kw] = xarray.DataArray(
    data=numpy.linspace(13.0, 45.0, 3, endpoint=True),
    dims=interval_kw,
    name=interval_kw,
    attrs={"units": "degC"}
)

interval_kw = "thresh_tasmax"
intervals[interval_kw] = xarray.DataArray(
    data=numpy.linspace(3.0, 27.0, 3, endpoint=True),
    dims=interval_kw,
    name=interval_kw,
    attrs={"units": "degC"}
)

interval_kw = "window"
intervals[interval_kw] = xarray.DataArray(
    data=numpy.arange(2, 5),
    dims=interval_kw,
    name=interval_kw
)

interval_kw = "thresh"
intervals[interval_kw] = xarray.DataArray(
    data=numpy.asarray((0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 10.0, 25.0, 50.0, 100.0)),
    dims=interval_kw,
    name=interval_kw,
    attrs={"units": "mmday"}
)

# Assign intervals to the parameters of every indicator.
for indicator_kw in indicators.keys():
    for parameter_kw in parameters[indicator_kw].keys():
        parameters[indicator_kw][parameter_kw] = intervals[parameter_kw]

In [ ]:
from braun.packages.xclim import compute_variations

In [ ]:
# Example: computations are perfomed for every indicator.

ds_hazard = {}
# Calculations are performed independently for each dataset to address
# the issue of different geographic coordinate systems as in step 5.
for dataset_kw in datasets_kw:
    ds_hazard[dataset_kw] = {}
    ds_remote = client.scatter(ds_daily[dataset_kw], broadcast=True)
    for indicator_kw, indicator in indicators.items():
        ds_hazard[dataset_kw][indicator_kw] = client.submit(
            compute_variations,
            indicator=indicator,
            variations=parameters[indicator_kw],
            ds=ds_remote,
            key=indicator_kw
        )
    ds_hazard[dataset_kw].update(client.gather(ds_hazard[dataset_kw]))
    del ds_remote  # Clean up memory.

## Step 7

Apply dimensionality reduction.
The aim is to condense information on the spatial and temporal dimensions to a single value for each hazard indicator. This scalar value quantifies the hazard driver represented by the indicator and is used for the risk calculation.

*Spatial dimensions are reduced by using EOF and keeping the first PC only. Temporal dimension is reduced by considering the sample mean.*

In [ ]:
import xeofs

In [ ]:
# Example: reduce spatial dimensions using EOF analysis.
# This process is very time consuming, even if already parallelised: it takes
# about 20 minutes to complete. Results are stored for convenience later. 

da_eof = {}
# Calculations are performed independently for each dataset to address
# the issue of different geographic coordinate systems as in step 5.
for dataset_kw in datasets_kw:
    da_eof[dataset_kw] = {}
    for indicator_kw, ds in ds_hazard[dataset_kw].items():
        eof = xeofs.models.EOF(n_modes=10, use_coslat=True, compute=False)
        eof.fit(ds, dim=("time", *parameters[indicator_kw].keys()))
        da_eof[dataset_kw][indicator_kw] = eof.scores(normalized=False).sel(mode=1)
        da_eof[dataset_kw][indicator_kw].assign_coords({"explained_variance_ratio": eof.explained_variance_ratio().sel(mode=1).values})
        da_eof[dataset_kw][indicator_kw].attrs.clear()  # Avoid issues with NetCDF format.

In [ ]:
# Example: reduce temporal dimension using sample mean.

da_hazard_scalar = {}
# Calculations are performed independently for each dataset to address
# the issue of different geographic coordinate systems as in step 5.
for dataset_kw in datasets_kw:
    da_hazard_scalar[dataset_kw] = {}
    for indicator_kw, ds in da_eof[dataset_kw].items():
        da_hazard_scalar[dataset_kw][indicator_kw] = da_eof[dataset_kw][indicator_kw].mean(dim="time")

## Step 8

Insert in Python information on the vulnerability of the system.

An xarray Dataset object is used to store values, with one of the coordinates which identify the assets composing the system. Each quantity, i.e. exposure, adaptive capacity and sensitivity, are contained each in a different xarray DataArray object.
The overall vulnerability can be evaluated from these quantities and stored as an additional DataArray object in the same xarray Dataset object.

*The system is composed by a given number of assets and they are referenced by a simple string containing an integer number. For ease consider only the resulting vulnerability as if it were evaluated from the other quantities and no dependence on time. Its values are drawn from the uniform distribution with real domain $[0, 1]$.*

In [ ]:
# Example: import assets in Python.

list_assets = [f"a_{n}" for n in range(1, 3)]

In [ ]:
# Example: insert information on vulnerability of assets.

ds_vulnerability = xarray.Dataset(
    data_vars={"vulnerability": ("asset", numpy.random.rand(len(list_assets)))},
    coords={"asset": list_assets}
)

## Step 9

Evaluate the risk matrix.
This matrix relates the vulnerability of each asset to every possibile hazard driver and quantify this relation.

Hazard drivers are quantified by indicators, hence for each asset the risk is evaluated by combining its vulnerability with the multidimensional object obtained for each indicator in step 7.
The resulting risk matrix is not just a bidimensional object but a multidimensional object with additional dimensions for each indicator parameter.

*Vulnerabilities and hazard indicators are combined through a product.*

In [ ]:
# Example: multiply the vulnerability of each asset with the hazard indicators.

def risk(hazard: xarray.DataArray, vulnerability: xarray.DataArray):
    _return = hazard * vulnerability  # Auto broadcast asset dimension as last.
    return _return

def func(da_remote, ds_remote, indicator_kw):
    da_out = risk(
        hazard=da_remote,
        vulnerability=ds_remote["vulnerability"]
    )
    da_out.name = f"{indicator_kw}_risk"
    return da_out

da_risk = {}
# Calculations are performed independently for each dataset to address
# the issue of different geographic coordinate systems as in step 5.
ds_remote = client.scatter(ds_vulnerability, broadcast=True)
for dataset_kw in datasets_kw:
    da_risk[dataset_kw] = {}
    for indicator_kw, indicator in indicators.items():
        da_remote = client.scatter(da_hazard_scalar[dataset_kw][indicator_kw])
        da_risk[dataset_kw][indicator_kw] = client.submit(
            func,
            da_remote,
            ds_remote,
            indicator_kw,
            key=indicator_kw
        )
        del da_remote  # Clean up memory.
    da_risk[dataset_kw].update(client.gather(da_risk[dataset_kw]))
del ds_remote  # Clean up memory.

In [ ]:
# Store data which takes long to evaluate.
c = []
for dataset_kw in datasets_kw:
    for indicator_kw in indicators.keys():
        c.append(ds_hazard[dataset_kw][indicator_kw].to_netcdf(
            path=pathlib.Path(dir_data, dataset_kw, f"{indicator_kw}.nc"),
            compute=False
        ))
        c.append(da_eof[dataset_kw][indicator_kw].to_netcdf(
            path=pathlib.Path(dir_data, dataset_kw, "eof", f"{indicator_kw}.nc"),
            compute=False
        ))
        c.append(da_risk[dataset_kw][indicator_kw].to_netcdf(
            path=pathlib.Path(dir_data, dataset_kw, "risk", f"{indicator_kw}.nc"),
            compute=False
        ))
dask.compute(c)

In [ ]:
# Close xarray objects to release memory.
for dataset_kw in datasets_kw:
    ds_hourly[dataset_kw].close()
    ds_daily[dataset_kw].close()
    ds_member[dataset_kw].close()
    for indicator_kw in indicators.keys():
        ds_hazard[dataset_kw][indicator_kw].close()
        da_eof[dataset_kw][indicator_kw].close()
        da_hazard_scalar[dataset_kw][indicator_kw].close()
        da_risk[dataset_kw][indicator_kw].close()
ds_vulnerability.close()